# 🚀 MorphoSpeech-LLM: Türkçe Otomatik Konuşma Tanıma (ASR) SOTA Eğitimi

Bu Google Colab notebook'u, **Türkçe Otomatik Konuşma Tanıma (ASR)** alanında akademik SOTA (State of the Art) sonuçları aşmak üzere tasarlanmıştır.

### 🎯 Hedefler:
1. Google/Meta **FLEURS Türkçe** ve Mozilla **Common Voice Türkçe** verisetlerinde SOTA (WER %6.8) altını hedeflemek.
2. **LoRA (PEFT)** ile **Whisper-Large-v3** veya **Whisper-Small** modelini Türkçe ek birleşimlerine (agglutinative morphology) uyarlamak.
3. Eğitim sonunda otomatik WER (%) ve CER (%) metriklerini hesaplayıp en iyi checkpoint'i kaydetmek.

> ⚠️ **Not:** Menüden **Çalışma Zamanı -> Çalışma zamanı türünü değiştir** seçeneğine girip **T4 GPU**, **L4 GPU** veya **A100 GPU** seçtiğinizden emin olun.

## 📦 Adım 1: Bağımlılıkların Yüklenmesi ve HF_TOKEN Ayarı

In [ ]:
# GPU Durumunu Kontrol Etme
!nvidia-smi

# Gerekli kütüphaneleri ve güncel peft/torchao kütüphanelerini yükleme
!pip install -q -U torchao peft torch torchaudio transformers 'datasets<3.0.0' evaluate jiwer librosa soundfile accelerate optuna morfessor sentencepiece

# Optional HF_TOKEN loading from Colab Secrets
import os
try:
    from google.colab import userdata
    token = userdata.get('HF_TOKEN')
    if token:
        os.environ['HF_TOKEN'] = token
        print('🔑 HF_TOKEN başarıyla aktifleştirildi!')
except Exception as e:
    print('ℹ️ HF_TOKEN olmadan anonim olarak devam ediliyor (Sorunsuz çalışır).')

## 🛠️ Adım 2: Reponun Klonlanması ve PYTHONPATH Ayarı

In [ ]:
import os, sys
if not os.path.exists('turkish-asr'):
    !git clone https://github.com/gokhanersoy/turkish-asr.git
    %cd turkish-asr
else:
    %cd turkish-asr
    !git pull

# Set PYTHONPATH to project root
sys.path.insert(0, os.getcwd())
os.environ['PYTHONPATH'] = os.getcwd()
print('✅ Proje dizini ve PYTHONPATH başarıyla ayarlandı:', os.getcwd())

## 🏋️‍♂️ Adım 3: Türkçe SOTA Fine-Tuning Eğitiminin Başlatılması

Colab GPU (A100 / L4 / T4) üzerinde **Whisper-Small** (veya `openai/whisper-large-v3`) modelini FLEURS Türkçe veriseti üzerinde eğitiyoruz.

In [ ]:
# Fine-Tuning Eğitimi (3 Epoch, LoRA PEFT Hizalandırma)
!PYTHONPATH=. python scripts/train_morphospeech.py \
    --model openai/whisper-small \
    --dataset fleurs \
    --train-samples 300 \
    --val-samples 50 \
    --epochs 3 \
    --batch-size 8 \
    --learning-rate 1e-4 \
    --output-dir checkpoints/morphospeech_sota

## 📊 Adım 4: Eğitilen SOTA Modelinin Karşılaştırmalı Değerlendirilmesi (WER & CER)

In [ ]:
# Eğitilen en iyi checkpoint üzerinde benchmark testi koşturma
!PYTHONPATH=. python scripts/run_benchmark.py \
    --model checkpoints/morphospeech_sota/best_sota_model \
    --dataset fleurs \
    --split test \
    --max-samples 50